# The Real Double Slit


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-20.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

An ideal double slit is two delta functions. A **real** one is two slits of finite
width $b$, separated by $a$, and that is not a new problem: it is a rect
**convolved** with two deltas.

$$f \otimes h = \textrm{rect}\left(\frac{x}{b}\right) \otimes
\left[\delta\left(x-\frac{a}{2}\right) + \delta\left(x+\frac{a}{2}\right)\right]$$

Convolving with a delta copies a function to that delta's position, so convolving one
slit with two deltas puts one slit at each. Then the convolution theorem does the rest:
a convolution in space is a product in frequency, so the far-field amplitude is the
single-slit sinc **times** the two-slit cosine.

That product is the whole result. The cosine supplies the fringes and the sinc supplies
the envelope that decides how bright each fringe is, which is why some orders in a real
double-slit pattern are missing entirely.

This is a **coherent** calculation throughout: every panel until the last is an
amplitude, and the last one squares it.


## Import libraries

NumPy and Matplotlib. `ConnectionPatch` draws the arrows that turn six axes into one
flow diagram, the same way `ch05-15` and `ch05-16` do.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch


In [ ]:
%matplotlib inline
# ==========================================
# 1. Configuration & Styling
# ==========================================
plot_params = {
    'font.size': 14,
    'font.family': 'sans-serif',
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.5,
}
plt.rcParams.update(plot_params)


def apply_custom_formatting(ax):
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.minorticks_on()
    ax.tick_params(which='both', direction="in", width=1)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=3.5)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')


## Define parameters

**Two lengths, and only two.** The slit width $b$ and the separation $a$. Everything in
both figures follows from their ratio, here $a/b = 3$, chosen so the slits are clearly
separated and a missing order lands somewhere visible.

**The transforms are analytic**, taken from the chapter's table:

$$\mathscr{F}\left\{\textrm{rect}\left(\frac{x}{b}\right)\right\} = b\,\textrm{sinc}
\left(\frac{kb}{2}\right), \qquad
\mathscr{F}\left\{\delta\left(x-\frac{a}{2}\right)+\delta\left(x+\frac{a}{2}\right)\right\}
= 2\cos\left(\frac{ka}{2}\right)$$

Note the unit weights on the deltas, which is what puts the factor of 2 in front of the
cosine. Some books give the deltas weight $1/2$ and drop that factor; the shape is
identical either way, and only the constant differs.

**Watch the sinc convention.** This text uses the *unnormalized*
$\textrm{sinc}(u) = \sin(u)/u$, while `np.sinc` is the *normalized*
$\sin(\pi z)/(\pi z)$, hence the helper that divides the argument by $\pi$.


In [ ]:
# ==========================================
# 2. Data Generation
# ==========================================
width1 = 1.0       # b, the width of one slit
sep1 = 3.0         # a, the separation between slit centers

N1 = 32768
L1 = 200.0
x1 = np.linspace(-L1 / 2, L1 / 2, N1, endpoint=False)
dx1 = x1[1] - x1[0]

# Angular spatial frequency, this text's convention
k1 = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(N1, d=dx1))


def unnormalized_sinc(u):
    """sinc(u) = sin(u)/u, this text's convention.

    np.sinc is the NORMALIZED sin(pi z)/(pi z), so divide the argument by pi.
    """
    return np.sinc(u / np.pi)


# --- Aperture plane ---
f_val1 = np.where(np.abs(x1) <= width1 / 2, 1.0, 0.0)      # one slit
# The deltas are drawn as arrows; for the numerical check the two slits are
# built directly, since a sampled delta is a spike of height 1/dx.
g_val1 = (np.where(np.abs(x1 - sep1 / 2) <= width1 / 2, 1.0, 0.0)
          + np.where(np.abs(x1 + sep1 / 2) <= width1 / 2, 1.0, 0.0))

# --- Far field, all analytic ---
F_val1 = width1 * unnormalized_sinc(k1 * width1 / 2)       # the single slit
H_val1 = 2 * np.cos(k1 * sep1 / 2)                         # the two deltas
G_val1 = F_val1 * H_val1                                   # their PRODUCT
envelope1 = 2 * width1 * unnormalized_sinc(k1 * width1 / 2)

# What a screen records
I_val1 = np.abs(G_val1) ** 2

# Characteristic spacings
fringe_k1 = 2 * np.pi / sep1        # spacing between fringes
env_zero1 = 2 * np.pi / width1      # first zero of the envelope


## Plot the grid

The same 2x3 flow diagram used throughout this chapter.

**The top row is the aperture plane.** One slit, two deltas, and their convolution:
two slits. Convolving with a delta is pure copying, so nothing is distorted, only
duplicated.

**The bottom row is the far field**, and it is where the work happens. The single slit
gives the broad sinc $F(k)$. The two deltas give the cosine $H(k)$, which oscillates
forever because a pair of deltas has no width to limit it. Their product is the fringe
pattern: **cosine fringes under a sinc envelope.**

The dashed curve in the bottom-right panel is that envelope, $\pm 2b\,\textrm{sinc}(kb/2)$.
It is the single-slit answer, and it caps every fringe. Where the envelope crosses zero
the fringes vanish no matter what the cosine is doing, which is what a **missing order**
is: a place where the two-slit interference would have produced a bright fringe but the
single-slit diffraction has already sent no light there.


In [ ]:
# ==========================================
# 3. Plotting the Grid
# ==========================================
fig1, axes1 = plt.subplots(2, 3, figsize=(16, 9))
plt.subplots_adjust(wspace=0.50, hspace=0.78, left=0.13, right=0.97,
                    top=0.90, bottom=0.08)

xlim1 = 3.6
klim1 = 20.0

# --- Top Row: the aperture plane ---
ax_f = axes1[0, 0]
ax_f.plot(x1, f_val1, 'r')
ax_f.fill_between(x1, 0, f_val1, color='red', alpha=0.15)
ax_f.set_title(r"$f(x) = \mathrm{rect}(x/b)$", fontsize=16, color='red')
ax_f.set_xlim(-xlim1, xlim1)
ax_f.set_ylim(-0.25, 1.5)
ax_f.set_xlabel(r"$x$")
ax_f.annotate("", xy=(-width1 / 2, 1.18), xytext=(width1 / 2, 1.18),
              arrowprops=dict(arrowstyle="<->", lw=1.5, color='red'))
ax_f.text(0, 1.24, r"$b$", ha='center', color='red', fontsize=14)
apply_custom_formatting(ax_f)

ax_h = axes1[0, 1]
# A delta has no finite height, so draw an arrow and label its weight
for sign1 in (+1, -1):
    ax_h.annotate("", xy=(sign1 * sep1 / 2, 1.0), xytext=(sign1 * sep1 / 2, 0),
                  arrowprops=dict(arrowstyle="-|>", lw=3, color='blue',
                                  shrinkA=0, shrinkB=0))
ax_h.text(sep1 / 2 + 0.18, 1.02, r"$1$", color='blue', fontsize=14)
ax_h.text(-sep1 / 2 - 0.18, 1.02, r"$1$", color='blue', fontsize=14, ha='right')
ax_h.axhline(0, color='black', linewidth=1, alpha=0.5)
ax_h.set_title(r"$h(x) = \delta(x - a/2) + \delta(x + a/2)$", fontsize=15,
               color='blue')
ax_h.set_xlim(-xlim1, xlim1)
ax_h.set_ylim(-0.25, 1.5)
ax_h.set_xlabel(r"$x$")
apply_custom_formatting(ax_h)

ax_g = axes1[0, 2]
ax_g.plot(x1, g_val1, 'g')
ax_g.fill_between(x1, 0, g_val1, color='green', alpha=0.15)
ax_g.set_title(r"$g(x) = f \otimes h$   (two real slits)", fontsize=16,
               color='green')
ax_g.set_xlim(-xlim1, xlim1)
ax_g.set_ylim(-0.25, 1.5)
ax_g.set_xlabel(r"$x$")
ax_g.annotate("", xy=(-sep1 / 2, 1.18), xytext=(sep1 / 2, 1.18),
              arrowprops=dict(arrowstyle="<->", lw=1.5, color='green'))
ax_g.text(0, 1.24, r"$a$", ha='center', color='green', fontsize=14)
apply_custom_formatting(ax_g)

# --- Bottom Row: the far field ---
ax_F = axes1[1, 0]
ax_F.plot(k1, F_val1, 'darkred')
ax_F.set_title(r"$F(k) = b\,\mathrm{sinc}(kb/2)$", fontsize=16, color='darkred')
ax_F.set_xlim(-klim1, klim1)
ax_F.set_ylim(-0.45, 1.35)
ax_F.set_xlabel(r"$k$")
apply_custom_formatting(ax_F)

ax_H = axes1[1, 1]
ax_H.plot(k1, H_val1, 'darkblue')
ax_H.set_title(r"$H(k) = 2\cos(ka/2)$", fontsize=16, color='darkblue')
ax_H.set_xlim(-klim1, klim1)
ax_H.set_ylim(-2.7, 2.7)
ax_H.set_xlabel(r"$k$")
apply_custom_formatting(ax_H)

ax_G = axes1[1, 2]
ax_G.plot(k1, envelope1, color='0.55', linestyle='--', linewidth=1.6)
ax_G.plot(k1, -envelope1, color='0.55', linestyle='--', linewidth=1.6)
ax_G.plot(k1, G_val1, 'darkgreen', linewidth=2.0)
ax_G.set_title(r"$G(k) = F \cdot H$", fontsize=16, color='darkgreen')
ax_G.set_xlim(-klim1, klim1)
ax_G.set_ylim(-2.7, 2.7)
ax_G.set_xlabel(r"$k$")
ax_G.text(0.97, 0.04, "dashed: the single-slit envelope",
          transform=ax_G.transAxes, fontsize=11, color='0.35',
          ha='right', va='bottom')
apply_custom_formatting(ax_G)

# ==========================================
# 4. Adding Flowchart Annotations
# ==========================================
fig1.canvas.draw()


def draw_arrow(ax_from, ax_to, text=""):
    """Vertical arrow between a column's two panels, as in ch05-10."""
    con = ConnectionPatch(xyA=(0.5, -0.32), xyB=(0.5, 1.14),
                          coordsA="axes fraction", coordsB="axes fraction",
                          axesA=ax_from, axesB=ax_to,
                          arrowstyle="-|>", shrinkB=5, linewidth=2.5,
                          color="black")
    ax_from.add_artist(con)
    if text:
        ax_to.text(0.56, 1.20, text, transform=ax_to.transAxes,
                   ha='left', va='bottom', fontsize=15)


draw_arrow(ax_f, ax_F, text="F.T.")
draw_arrow(ax_h, ax_H, text="F.T.")
draw_arrow(ax_g, ax_G, text="F.T.")


def gutter(ax_left, ax_right):
    """Center of the CLEAR gap between two side-by-side panels.

    The right panel's y-tick labels stick out to the left of its axes box, so
    centering between the boxes alone drops the glyph on top of them. Measuring
    where those labels actually begin gives a gap that is genuinely empty, and
    removes the need for a hand-tuned nudge.
    """
    fig = ax_left.figure
    renderer = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    right_x0 = ax_right.get_position().x0
    for lab in ax_right.get_yticklabels():
        if lab.get_visible() and lab.get_text():
            right_x0 = min(
                right_x0,
                lab.get_window_extent(renderer).transformed(inv).x0)
    return (ax_left.get_position().x1 + right_x0) / 2


def row_mid(ax):
    p = ax.get_position()
    return (p.y0 + p.y1) / 2


gx1, gx2 = gutter(ax_f, ax_h), gutter(ax_h, ax_g)
y_top, y_bot = row_mid(ax_f), row_mid(ax_F)

# One x for both the operator word and its symbol, so they stay aligned.
op_x = gx1

fig1.text(op_x, y_top + 0.055, "convolve", fontsize=12, ha='center', va='center')
fig1.text(op_x, y_top, r"$\otimes$", fontsize=32, ha='center', va='center')
fig1.text(gx2, y_top, r"$=$", fontsize=32, ha='center', va='center')

fig1.text(op_x, y_bot + 0.055, "multiply", fontsize=12, ha='center', va='center')
fig1.text(op_x, y_bot, r"$\times$", fontsize=32, ha='center', va='center')
fig1.text(gx2, y_bot, r"$=$", fontsize=32, ha='center', va='center')

fig1.text(0.035, y_top, "Aperture\nPlane", fontsize=17, weight='bold',
          rotation=90, va='center', ha='center')
fig1.text(0.035, y_bot, "Far Field\n(amplitude)", fontsize=17, weight='bold',
          rotation=90, va='center', ha='center')

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.


In [ ]:
# fig1.savefig("../figures/ch05/ch05-20-real-double-slit.jpg", dpi=200, bbox_inches='tight')


## What the screen actually shows

Every panel above is an **amplitude**. A screen does not record amplitude; it records
irradiance, the magnitude squared. So the pattern you would photograph is

$$I(k) \;\propto\; \left\lvert G(k) \right\rvert^2 \;=\;
\left\lvert 2b\,\textrm{sinc}\left(\frac{kb}{2}\right)
\cos\left(\frac{ka}{2}\right) \right\rvert^2 \;=\;
4b^2\,\textrm{sinc}^2\left(\frac{kb}{2}\right)\cos^2\left(\frac{ka}{2}\right)$$

which is the expression the question quotes, with the constant $4b^2$ absorbed into
$I_0$. The figure below plots both and they lie on top of each other.

Squaring changes the picture in two ways worth noticing. Every fringe becomes
**positive**, since a magnitude squared cannot be negative, so the negative lobes of the
amplitude fold up into bright fringes. And the contrast increases: the envelope's weak
tails are suppressed harder than its peak, because squaring a number smaller than one
makes it smaller still.

This is the same step, and the same reason, as the one that turns the amplitude PSF $h$
into the intensity PSF $\left\lvert h \right\rvert^2$ elsewhere in this chapter.


In [ ]:
# ==========================================
# 5. What the screen records
# ==========================================
fig2, ax2 = plt.subplots(figsize=(13, 6))

# The closed form the question quotes, for comparison
closed_form2 = (4 * width1 ** 2
                * unnormalized_sinc(k1 * width1 / 2) ** 2
                * np.cos(k1 * sep1 / 2) ** 2)

ax2.plot(k1, closed_form2, color='orange', linewidth=5, alpha=0.55,
         label=r'$4b^2\,\mathrm{sinc}^2(kb/2)\cos^2(ka/2)$')
ax2.plot(k1, I_val1, 'darkgreen', linewidth=2.0,
         label=r'$\left|G(k)\right|^2$ from the figure above')
ax2.plot(k1, envelope1 ** 2, color='0.55', linestyle='--', linewidth=1.6,
         label='single-slit envelope, squared')
ax2.fill_between(k1, 0, I_val1, color='green', alpha=0.10)

# The fringe spacing
ax2.annotate("", xy=(0, 4.45), xytext=(fringe_k1, 4.45),
             arrowprops=dict(arrowstyle="<->", lw=1.6, color='darkgreen'))
ax2.text(fringe_k1 / 2, 4.55, r"$2\pi/a$", ha='center', color='darkgreen',
         fontsize=15)

# The missing order: an envelope zero landing on a cosine maximum
order2 = env_zero1 / fringe_k1
ax2.annotate("missing order:\nthe envelope is zero\nwhere a fringe would be",
             xy=(env_zero1, 0.02), xytext=(env_zero1 + 1.0, 1.9),
             arrowprops=dict(arrowstyle="->", color='darkred', linewidth=1.8),
             fontsize=12, color='darkred', ha='left')

ax2.set_title("Irradiance on the screen: the amplitude, squared", fontsize=17)
ax2.set_xlabel(r"$k$")
ax2.set_ylabel(r"$I \propto \left|G\right|^2$")
ax2.set_xlim(-20, 20)
ax2.set_ylim(-0.3, 5.2)
ax2.legend(loc='upper right', frameon=True, edgecolor='black', fancybox=False,
           fontsize=12)
apply_custom_formatting(ax2)

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.


In [ ]:
# fig2.savefig("../figures/ch05/ch05-20-double-slit-intensity.jpg", dpi=200, bbox_inches='tight')
